# U-Net input ablations: Ddc / full Yb / imaginary Yb / real Yb

Updated interface for `dl_normalized_datasets` v2. Set CASE=30 or 57. The terminal runner and notebook use the same code. Run from this code folder.


In [3]:
from pathlib import Path
import subprocess
import sys
import json
import csv

CODE_DIR = Path.cwd()  # Folder containing train_ablation.py and updated model files
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 30              # 30, 39 or 57
TARGET = 'dq'          # 'dq' or 'vmag'; one target type per run
MODELS = ['unet', 'attention_unet']
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real']
SEEDS = [42, 123, 2026]
EPOCHS = 100
BATCH_SIZE = 64
LOSS_ALPHA = 0.5       # 1.0 means pure MSE
DEVICE = 'cuda'
SMOKE = False         # True uses tiny subsets/two epochs; not a performance estimate
RESUME = False
AMP = True

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}'
assert (CODE_DIR / 'train_ablation.py').is_file(), 'Set CODE_DIR to the extracted updated code folder'
for name in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{name}.npz').is_file(), f'Missing {name} data'
print('Data:', DATA_DIR)
print('Results:', OUT_DIR)


Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case30_dq


In [4]:
command = [sys.executable, '-u', str(CODE_DIR / 'train_ablation.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--models', *MODELS, '--variants', *VARIANTS,
    '--target', TARGET, '--seeds', *map(str, SEEDS),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--loss-alpha', str(LOSS_ALPHA), '--device', DEVICE]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
if not AMP: command.append('--no-amp')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/train_ablation.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case30_dq --models unet attention_unet --variants ddc ddc_yb ddc_imag ddc_real --target dq --seeds 42 123 2026 --epochs 100 --batch-size 64 --loss-alpha 0.5 --device cuda
PyTorch 2.10.0+cu128 device cuda
Dataset(dq, X=(6880, 3, 30, 30), y=(6880, 1, 30, 30)) 
 Dataset(dq, X=(1474, 3, 30, 30), y=(1474, 1, 30, 30))

unet ddc seed=42: 1,927,009 parameters; AMP=True
epoch=  1 loss=3.39281 val_R2=0.7616608844608191 val_RMSE=2.083983 (19.2s)
epoch=  2 loss=0.22622 val_R2=0.7531411608194515 val_RMSE=2.120903 (1.0s)
epoch=  3 loss=0.19181 val_R2=0.8441564665492358 val_RMSE=1.685159 (0.9s)
epoch=  4 loss=0.17760 val_R2=0.8531830566641778 val_RMSE=1.635628 (0.9s)
epoch=  5 loss=0.16723 val_R2=0.85

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/train_ablation.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case30_dq', '--models', 'unet', 'attention_unet', '--variants', 'ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', '--target', 'dq', '--seeds', '42', '123', '2026', '--epochs', '100', '--batch-size', '64', '--loss-alpha', '0.5', '--device', 'cuda'], returncode=0)

In [5]:
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Metrics use original units:', summary['protocol']['metric_units'])
for model, variants in summary['aggregate'].items():
    for variant, splits in variants.items():
        print(model, variant, 'test:', splits['test'])
print('Training-mean baseline:', summary['training_mean_baseline'])
print('Individual runs:', OUT_DIR / 'comparison.csv')


Metrics use original units: raw DQ
unet ddc test: {'r2': {'mean': 0.8670717375430609, 'std_across_seeds': 0.002585084380000269}, 'rmse': {'mean': 1.5712074871799668, 'std_across_seeds': 0.0152363670334901}, 'mae': {'mean': 0.6567481402168446, 'std_across_seeds': 0.007244651339698103}}
unet ddc_yb test: {'r2': {'mean': 0.8665581232587874, 'std_across_seeds': 0.0016828668623621635}, 'rmse': {'mean': 1.5742684157782743, 'std_across_seeds': 0.00994489291368569}, 'mae': {'mean': 0.6871942323510819, 'std_across_seeds': 0.034089991042974485}}
unet ddc_imag test: {'r2': {'mean': 0.8672939251008988, 'std_across_seeds': 0.0015188991267235318}, 'rmse': {'mean': 1.5699258864822647, 'std_across_seeds': 0.008983491641823329}, 'mae': {'mean': 0.6680864923611151, 'std_across_seeds': 0.027586392804819587}}
unet ddc_real test: {'r2': {'mean': 0.8680779567725123, 'std_across_seeds': 0.0014540872352318114}, 'rmse': {'mean': 1.5652827173229678, 'std_across_seeds': 0.008613023624981243}, 'mae': {'mean': 0.6

## Notes

All four variants share the same fitted training normalization, parameter count within each architecture, initial weights per seed, and training protocol. Excluded slots are masked after normalization. Test data is evaluated only after validation checkpoint selection. Do not apply a second scaler or sigmoid to standardized targets. The saved predictions and R2/RMSE/MAE are in original units. See README.md for inference and recovery.
